# Cross-Session Population Analysis Summary

This notebook loads population analysis results (generated by population_analysis.ipynb) from multiple imaging sessions across animals and performs statistical comparisons and visualizations.

Last edited: WMS 3/13/26

In [ ]:
# Imports
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from statsmodels.stats.multitest import multipletests

from session_config import *
from xsession_utils import *

#### define data path and determine whether to save out files

In [ ]:
respath = r"C:\miniscope_files\analyzed\xsessionxanimal_results\population_metrics"

In [ ]:
save_csvs=False

In [ ]:
save_figs=False

In [ ]:
save_stats=False

#### define decoding types

In [ ]:
# Decoding comparison definitions
DECODING_TYPES = {
    'location_empty1': {
        'filename': 'location_empty1_decoding_res.pkl',
        'label': 'Location (Empty)',
        'short_label': 'Loc-E1',
        'color': '#545454'
    },
    'location_empty1_within': {
        'filename': 'location_empty1_decoding_res_within_animal.pkl',
        'label': 'Location (Empty)',
        'short_label': 'Loc-E1',
        'color': '#545454'
    },
    'location_empty1_across': {
        'filename': 'location_empty1_decoding_res_across_animals.pkl',
        'label': 'Location (Empty)',
        'short_label': 'Loc-E1',
        'color': '#545454'
    },
    'location_social': {
        'filename': 'location_social_trials_decoding_res.pkl',
        'label': 'Location (Social)',
        'short_label': 'Loc-Soc',
        'color': '#2E86AB'
    },
    'location_social_within': {
        'filename': 'location_social_trials_decoding_res_within_animal.pkl',
        'label': 'Location (Social)',
        'short_label': 'Loc-Soc',
        'color': '#2E86AB'
    },
    'location_social_across': {
        'filename': 'location_social_trials_decoding_res_across_animals.pkl',
        'label': 'Location (Social)',
        'short_label': 'Loc-Soc',
        'color': '#2E86AB'
    },
    'identity': {
        'filename': 'identity_decoding_res.pkl',
        'label': 'Identity',
        'short_label': 'Identity',
        'color': '#A23B72'
    },
    'identity_within': {
        'filename': 'identity_decoding_res_within_animal.pkl',
        'label': 'Identity',
        'short_label': 'Identity',
        'color': '#A23B72'
    },
    'identity_across': {
        'filename': 'identity_decoding_res_across_animals.pkl',
        'label': 'Identity',
        'short_label': 'Identity',
        'color': '#A23B72'
    }
}

# Metric options for decoding results
METRIC_OPTIONS = ['accuracy', 'f1', 'auroc']

# Null distribution options
NULL_OPTIONS = {
    'circular': 'null_shift_results',
    'circular_aware': 'null_shift_aware_results', 
    'shuffle': 'null_shuffle_results'
}

In [ ]:
for dt in DECODING_TYPES:
    print(dt)

#### Load data

Population-level metrics

In [ ]:
all_pop_data = load_all_population_data(ANIMAL_PATHS,DECODING_TYPES)

In [ ]:
#check the proper ordering of paths if you like
animal_id = '8946'
session_paths=ANIMAL_PATHS[animal_id]
for session_idx, path in enumerate(session_paths):
    print(f'{session_idx}:{path}')

Partner preference scores

In [ ]:
PPT_scores_path = r"\\penc8.rc.int.colorado.edu\DonaldsonLab\Sheeran\experiments\hippocampus miniscope imaging\imaging animals ppt results\imaging_animal_ppt_scores.xlsx"

In [ ]:
good_imaging_animals = [k for k, v in ANIMAL_PATHS.items()]
good_imaging_animals

In [ ]:
#loading and formatting
imaging_ppt_scores = pd.read_excel(PPT_scores_path,sheet_name=0)
imaging_ppt_scores['Animal']=imaging_ppt_scores['Animal'].astype(str)
imaging_ppt_scores_good = imaging_ppt_scores[imaging_ppt_scores['Animal'].isin(good_imaging_animals)].copy().reset_index(drop=True)
imaging_ppt_scores_good

In [ ]:
tidy_ppt_scores = make_ppt_lookup(imaging_ppt_scores_good, score_types=('cup', 'chamber'))
tidy_ppt_scores

#### extract sparsity data

In [ ]:
sparsity_path = os.path.join(respath,'sparsity_metrics')

plotting functions

In [ ]:
def plot_sparsity_distribution(df, metric='median_rate', session_indices=None,
                                figsize=(8, 5), save_path=None):
    """
    Plot distribution of sparsity metrics across sessions.
    
    Parameters:
    -----------
    metric : str
        'median_rate', 'mean_rate' for event rates
        'prop_X_pts_empty' for empty pseudotrial proportions
    """
    if session_indices is None:
        session_indices = SESSION_SUBSETS['social']
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Get colors for sessions
    palette = {SESSION_CONFIG['short_labels'][i]: SESSION_CONFIG['colors'][i] 
               for i in session_indices}
    
    # Create strip plot with jitter
    sns.stripplot(data=df, x='short_label', y='value', hue='short_label',
                  palette=palette, size=10, alpha=0.7, ax=ax, legend=False)
    
    # Add mean bars
    session_order = [SESSION_CONFIG['short_labels'][i] for i in session_indices]
    for i, session in enumerate(session_order):
        session_vals = df[df['short_label'] == session]['value']
        if len(session_vals) > 0:
            mean_val = session_vals.mean()
            ax.hlines(mean_val, i - 0.2, i + 0.2, colors='black', linewidth=2)
    
    # Labels
    if 'rate' in metric:
        ax.set_ylabel('Median Event Rate (events/sec)', fontsize=12)
    else:
        ax.set_ylabel('Proportion Empty Pseudotrials', fontsize=12)
    
    ax.set_xlabel('Session', fontsize=12)
    ax.set_title(f'{metric}', fontsize=14)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    return fig, ax


Median calcium event rates per animal per session

In [ ]:
median_rates_df = extract_sparsity_metric(all_pop_data,'median_rate',session_indices=SESSION_SUBSETS['social'])
median_rates_df

In [ ]:
median_event_rates_path = os.path.join(sparsity_path,"median_event_rates_by_session.csv")
if save_csvs == True:
    median_rates_df.to_csv(median_event_rates_path,index=False)

In [ ]:
median_event_rate_plot_path = os.path.join(sparsity_path,"median_event_rates_by_session.png")
if save_figs == True:
    event_rate_save_path = median_event_rate_plot_path
else:
    event_rate_save_path = None

In [ ]:
fig, ax = plot_sparsity_distribution(median_rates_df,metric='median_rate',session_indices=None,save_path=event_rate_save_path)

In [ ]:
event_rate_stats = friedmantest_across_sessions(median_rates_df)
event_rate_stats

Proportion of 1 second pseudotrials from social trials that are 'empty' (i.e., do not contain a single calcium event)

In [ ]:
#extract all social pseudotrial metrics
percent_empty_pr_pseudotrials = extract_sparsity_metric(all_pop_data,'prop_pr_pts_empty',session_indices=SESSION_SUBSETS['social'])
percent_empty_pl_pseudotrials = extract_sparsity_metric(all_pop_data,'prop_pl_pts_empty',session_indices=SESSION_SUBSETS['social'])
percent_empty_nr_pseudotrials = extract_sparsity_metric(all_pop_data,'prop_nr_pts_empty',session_indices=SESSION_SUBSETS['social'])
percent_empty_nl_pseudotrials = extract_sparsity_metric(all_pop_data,'prop_nl_pts_empty',session_indices=SESSION_SUBSETS['social'])
percent_empty_l_e1_pseudotrials = extract_sparsity_metric(all_pop_data,'prop_l_e1_pts_empty',session_indices=SESSION_SUBSETS['social'])
percent_empty_r_e1_pseudotrials = extract_sparsity_metric(all_pop_data,'prop_r_e1_pts_empty',session_indices=SESSION_SUBSETS['social'])

percent_empty_pseudotrials_all = pd.concat([percent_empty_pr_pseudotrials,percent_empty_pl_pseudotrials,
                                            percent_empty_nr_pseudotrials,percent_empty_nl_pseudotrials,
                                            percent_empty_l_e1_pseudotrials,percent_empty_r_e1_pseudotrials]).reset_index(drop=True)

percent_empty_pseudotrials_all

In [ ]:
prop_empty_pts_path = os.path.join(sparsity_path,"percent_empty_pseudotrials_by_session.csv")
if save_csvs == True:
    percent_empty_pseudotrials_all.to_csv(prop_empty_pts_path,index=False)

-------------------

### Decoding

#### extract decoding data

will create two types of dataframe - one dataframe for each metric across all animals and all sessions, and one dataframe for each session across all animals and metrics

In [ ]:
decoding_path = os.path.join(respath,'decoding')
decoding_path

one df for each decoding variable across all animals/sessions:

In [ ]:
all_decode_types_across_sessions = {}
for dt in DECODING_TYPES.keys():
    all_decode_types_across_sessions[dt]=extract_decoding_metric(all_pop_data,decode_type=dt,null_options=NULL_OPTIONS,metric='accuracy',
                                                                 session_indices=SESSION_SUBSETS['social'],animals=None,include_null=True,
                                                                 null_type='circular_aware')

In [ ]:
#can call each metric's results as a df
all_decode_types_across_sessions['location_social']

In [ ]:
across_sesh_path = os.path.join(decoding_path,'each_metric_across_sessions')
if save_csvs==True:
    for dt in DECODING_TYPES.keys():
        this_metric_across_sesh_path = os.path.join(across_sesh_path,f"{dt}_all_animals_all_sessions.csv")
        all_decode_types_across_sessions[dt].to_csv(this_metric_across_sesh_path,index=False)

one df for each session across all animals and variables

In [ ]:
each_session_all_decode_types = {}
for sesh_idx in SESSION_SUBSETS['social']:
    each_session_all_decode_types[sesh_idx]=extract_all_decoding_types(all_pop_data,decoding_types=DECODING_TYPES,metric='accuracy',
                                                                       session_idx=sesh_idx,animals=None,null_type='circular_aware',
                                                                       null_options=NULL_OPTIONS)

In [ ]:
#can call each sessions's results as a df
each_session_all_decode_types[1]

In [ ]:
within_sesh_path = os.path.join(decoding_path,'each_session_all_metrics')
if save_csvs==True:
    for sesh_idx in SESSION_SUBSETS['social']:
        all_metrics_this_sesh_path = os.path.join(within_sesh_path,f"{SESSION_CONFIG['short_labels'][sesh_idx]}_all_animals_all_metrics.csv")
        each_session_all_decode_types[sesh_idx].to_csv(all_metrics_this_sesh_path,index=False)

#### How does decoding performance vary as a function of the # of bins (pseudotrials) the decoder had access to train on?

We want to be able to use SVM performance as a measure of how well our recorded population can discriminate between two variables under different experimental conditions. But a major concern for this type of analysis is that SVM performance is influenced by the dimensionality of the input data. For our purposes, dimensionality can either mean the # of neurons (features) or the # of samples (population vectors from different behavioral bins/pseudobouts). We already controlled for the former by running one run for each decoding variable with the lowest # of neurons across sessions within an animal (the "_within" results) and another run with the lowest # of neurons across animals and session (the "_across" variable). This section examines the impact of the # of samples (pseudobouts) available for any given decoder run to determine whether we need to drop outlier sessions when performing our cross-session or cross-animal comparisons.

In [ ]:
pt_vs_decoding_path = os.path.join(decoding_path,'pseudotrials_vs_decoding_performance')

Step 1: define plotting code

In [ ]:
def plot_decoding_vs_pts(all_data, null_options=NULL_OPTIONS, min_pseudotrials=None, decode_type='identity', metric='accuracy',
                          session_indices=None, figsize=(6, 5), save_path=None):
    """
    Plot decoding performance vs number of pseudotrials available.
    """
    if session_indices is None:
        session_indices = SESSION_SUBSETS['social']
    
    df = extract_decoding_metric(all_data, decode_type, null_options=null_options, metric=metric,
                                  session_indices=session_indices) #animals=None, include_null=True, null_type='circular_aware'
    df = df.dropna(subset=['true_value', 'mean_train_pts_per_iter']) #total_pts

    if min_pseudotrials is not None:
        df = df[df['mean_train_pts_per_iter']>=min_pseudotrials].copy().reset_index(drop=True)

    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Color by session
    for s_idx in session_indices:
        s_data = df[df['session_idx'] == s_idx]
        color = SESSION_CONFIG['colors'][s_idx]
        label = SESSION_CONFIG['short_labels'][s_idx]
        ax.scatter(s_data['mean_train_pts_per_iter'], s_data['true_value'], 
                  c=color, s=80, alpha=0.7, label=label, edgecolors='white')
    
    # Correlation
    if len(df) > 2:
        r, p = stats.pearsonr(df['mean_train_pts_per_iter'], df['true_value'])
        ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.3f}', 
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Regression line
        z = np.polyfit(df['mean_train_pts_per_iter'], df['true_value'], 1)
        p_line = np.poly1d(z)
        x_line = np.linspace(df['mean_train_pts_per_iter'].min(), df['mean_train_pts_per_iter'].max(), 100)
        ax.plot(x_line, p_line(x_line), 'k--', alpha=0.5, linewidth=1)
    
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)
    
    ax.set_xlabel('Mean # of trained pseudotrials per decoder iteration', fontsize=12)
    metric_labels = {'accuracy': 'Decoding Accuracy', 'f1': 'F1 Score', 'auroc': 'AUROC'}
    ax.set_ylabel(metric_labels.get(metric, metric), fontsize=12)
    
    decode_label = DECODING_TYPES[decode_type]['label']
    if min_pseudotrials is not None:
        ax.set_title(f'{decode_label} decoding performance vs. amount of training data available to decoder (min. {min_pseudotrials} bins)', fontsize=14)
    else:
        ax.set_title(f'{decode_label} decoding performance vs. amount of training data available to decoder (all bins)', fontsize=14)
    
    ax.legend(loc='lower right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    return fig, ax



Step 2: Define stats functions (Pearson correlation coefficient across all sessions/animals, or PCC within sessions with Benjamini-Hochberg correction across sessions)

In [ ]:
def correlate_across_within_sessions(df, x_col, y_col, comparison='across_sessions', session_indices=None,
                              min_pseudotrials=None, min_n=3, alpha=0.05):
    """
    Compute Pearson correlations between two columns within each session or across sessions.
    Decoding-specific function (see xsession_utils.py for a more generic version)
    
    Parameters
    ----------
    df : pd.DataFrame
        Likely an ouput of extract_decoding_metric(). Must contain 'session_idx', 'decode_type', x_col, and y_col columns.
    x_col : str
        Column name for the predictor (e.g. 'mean_train_pts_per_iter' or 'ppref_score_cup').
    y_col : str
        Column name for the outcome (e.g. 'true_value').
    comparison : str
        'across_sessions' or 'within_sessions'. Indicates whether to pool data from across all sessions to perform one Pearson's correlation 
        or to leave them segregated by session and to perform a Pearson's correlation on each session individually. Will also perform a Benjamini-Hochberg
        correction for multiple comparisons if the latter.
    session_indices : list[int] or None
        Sessions to include. If None, uses all unique sessions in df.
    min_pseudotrials : float or None
        If provided, exclude rows with mean_train_pts_per_iter below this value.
    min_n : int
        Minimum number of valid data points per session to run correlation.

    Returns
    -------
    pd.DataFrame
        One row per session with columns: session_idx, short_label, n, r, p_value,
        x_mean, y_mean.
    """
    if min_pseudotrials is not None:
        df = df[df['mean_train_pts_per_iter'] >= min_pseudotrials].copy()

    if session_indices is None:
        session_indices = sorted(df['session_idx'].unique())

    rows = []
    data = df.dropna(subset=[x_col, y_col])
    data = data[data['session_idx'].isin(session_indices)]
    decoding_type_info = data['decode_type']
    if len(decoding_type_info.unique())>1:
        raise ValueError("Input dataframe must only have one decoding variable!")
    
    if comparison=='across_sessions':
        row = {'decoding_type':decoding_type_info.values[0]}
        row['n'] = len(data)
        if len(data) >= min_n:
            r, p = stats.pearsonr(data[x_col], data[y_col])
            row['r'] = r
            row['p_value'] = p
        else:
            row['r'] = np.nan
            row['p_value'] = np.nan
            print(f"n={len(data)}, too few for correlation (need >= {min_n})")
        row[f'{x_col}_mean'] = data[x_col].mean() if len(data) > 0 else np.nan
        row[f'{x_col}_std'] = data[x_col].std() if len(data) > 0 else np.nan
        row[f'{y_col}_mean'] = data[y_col].mean() if len(data) > 0 else np.nan
        row[f'{y_col}_std'] = data[y_col].std() if len(data) > 0 else np.nan
        rows.append(row)
        
    elif comparison=='within_sessions':
        for s_idx in session_indices:
            row = {'decoding_type':decoding_type_info.values[0]}
            s_data = data[data['session_idx'] == s_idx]
    
            row['session_idx']=s_idx
            row['short_label']=SESSION_CONFIG['short_labels'][s_idx]
            row['n']=len(s_data)

            if len(s_data) >= min_n:
                r, p = stats.pearsonr(s_data[x_col], s_data[y_col])
                row['r'] = r
                row['p_value'] = p
            else:
                row['r'] = np.nan
                row['p_value'] = np.nan
                print(f"  {row['short_label']}: n={len(s_data)}, too few for correlation (need >= {min_n})")

            row[f'{x_col}_mean'] = s_data[x_col].mean() if len(s_data) > 0 else np.nan
            row[f'{x_col}_std'] = s_data[x_col].std() if len(s_data) > 0 else np.nan
            row[f'{y_col}_mean'] = s_data[y_col].mean() if len(s_data) > 0 else np.nan
            row[f'{y_col}_std'] = s_data[y_col].std() if len(s_data) > 0 else np.nan
            rows.append(row)
            
    else:
        raise ValueError(f"comparison must be 'across_sessions' or 'within_sessions', got {comparison}")

    summary_df = pd.DataFrame(rows)
    if comparison=='within_sessions': #Benjamini-Hochberg correction if multiple comparisons
        valid_mask = summary_df['p_value'].notna()
        if valid_mask.sum() > 0:
            rejected, p_corrected, _, _ = multipletests(
                summary_df.loc[valid_mask, 'p_value'], alpha=alpha, method='fdr_bh')
            summary_df['p_fdr'] = np.nan
            summary_df.loc[valid_mask, 'p_fdr'] = p_corrected
            summary_df['significant'] = False
            summary_df.loc[valid_mask, 'significant'] = rejected
        summary_df = summary_df.rename(columns={'p_value':'p_uncorrected'})

    return summary_df

Step 3: Call the dfs we'll work with -- decoding performance across sessions, holding # of neurons constant across animals/sessions (n=33 as that was the minimum for any session across all animals)

In [ ]:
identity_across_all_sessions = all_decode_types_across_sessions['identity_across'].copy()
social_location_across_all_sessions = all_decode_types_across_sessions['location_social_across'].copy()
empty_location_across_all_sessions = all_decode_types_across_sessions['location_empty1_across'].copy()
identity_across_all_sessions

Step 4: Plot # of pseudotrials vs. decoding performance for each metric, all sessions/all animals (holding # of neurons constant across points) and run correlation stats *with or without* dropping sessions that fall below a minimum pseudotrial threshold

In [ ]:
#choosing these values bc it has empirically seemed to be the best balance between minimizing correlation bw decoding performance and # of pseudotrials 
#vs. still having enough sessions to enable comparisons

min_pseudotrials_per_iter_social = 25
min_pseudotrials_per_iter_empty = 10

In [ ]:
#save out information about the sessions whose data was dropped for not passing the minimum pseudotrial threshold

too_few_pts_identity = identity_across_all_sessions[identity_across_all_sessions['mean_train_pts_per_iter']<min_pseudotrials_per_iter_social]
too_few_pts_social_location = social_location_across_all_sessions[social_location_across_all_sessions['mean_train_pts_per_iter']<min_pseudotrials_per_iter_social]
too_few_pts_empty_location = empty_location_across_all_sessions[empty_location_across_all_sessions['mean_train_pts_per_iter']<min_pseudotrials_per_iter_empty]

too_few_pts = [too_few_pts_identity,too_few_pts_social_location,too_few_pts_empty_location]
pts_short = []
for pt in too_few_pts:
    pt_short = pt.loc[:,['animal','session_idx','session_label','decode_type','total_pts','mean_train_pts_per_iter']]
    pts_short.append(pt_short)
pts_short_df = pd.concat(pts_short)
pts_short_df

In [ ]:
too_few_pts_dropped_session_info_path = os.path.join(pt_vs_decoding_path,'decoding_runs_below_training_data_threshold.csv')
if save_csvs:
    pts_short_df.to_csv(too_few_pts_dropped_session_info_path,index=False)

Identity, all sessions

In [ ]:
if save_figs:
    save_path_identity_all_pts = os.path.join(pt_vs_decoding_path,'identity_decoding_vs_num_pseudotrials_all_sessions.svg')
else:
    save_path_identity_all_pts = None

In [ ]:
fig, ax = plot_decoding_vs_pts(all_pop_data, null_options=NULL_OPTIONS, min_pseudotrials=None, decode_type='identity_across', metric='accuracy',
                          session_indices=SESSION_SUBSETS['social'], figsize=(6, 5), save_path=save_path_identity_all_pts)

In [ ]:
#stats
identity_vs_pts_all_pts = correlate_across_within_sessions(
    identity_across_all_sessions, x_col='mean_train_pts_per_iter', y_col='true_value',
    comparison='across_sessions',session_indices=SESSION_SUBSETS['social'],min_pseudotrials=None
)
identity_vs_pts_all_pts

Identity, only sessions with at least 25 pseudotrials per training dataset

In [ ]:
if save_figs:
    save_path_identity_enough_pts = os.path.join(pt_vs_decoding_path,'identity_decoding_vs_num_pseudotrials_enough_pseudotrials.svg')
else:
    save_path_identity_enough_pts = None

In [ ]:
fig, ax = plot_decoding_vs_pts(all_pop_data, null_options=NULL_OPTIONS, min_pseudotrials=min_pseudotrials_per_iter_social, decode_type='identity_across', metric='accuracy',
                          session_indices=SESSION_SUBSETS['social'], figsize=(6, 5), save_path=save_path_identity_enough_pts)

In [ ]:
#stats
identity_vs_pts_enough_pts = correlate_across_within_sessions(
    identity_across_all_sessions, x_col='mean_train_pts_per_iter', y_col='true_value',
    comparison='across_sessions',session_indices=SESSION_SUBSETS['social'],min_pseudotrials=min_pseudotrials_per_iter_social
)
identity_vs_pts_enough_pts

In [ ]:
identity_vs_pts_all_pts['min_mean_pts_per_iter']=None
identity_vs_pts_enough_pts['min_mean_pts_per_iter']=min_pseudotrials_per_iter_social

identity_vs_pts_compare = pd.concat([identity_vs_pts_all_pts,identity_vs_pts_enough_pts])
identity_vs_pts_compare

In [ ]:
save_path_identity_decoding_vs_pts = os.path.join(pt_vs_decoding_path,'identity_decoding_vs_num_training_pseudotrials.csv')
if save_csvs:
    identity_vs_pts_compare.to_csv(save_path_identity_decoding_vs_pts,index=None)

Location during the social trials, all sessions

In [ ]:
if save_figs:
    save_path_social_location_all_pts = os.path.join(pt_vs_decoding_path,'social_location_decoding_vs_num_pseudotrials_all_sessions.svg')
else:
    save_path_social_location_all_pts = None

In [ ]:
fig, ax = plot_decoding_vs_pts(all_pop_data, null_options=NULL_OPTIONS, min_pseudotrials=None, decode_type='location_social_across', metric='accuracy',
                          session_indices=SESSION_SUBSETS['social'], figsize=(6, 5), save_path=save_path_social_location_all_pts)

In [ ]:
#stats
social_location_vs_pts_all_pts = correlate_across_within_sessions(
    social_location_across_all_sessions, x_col='mean_train_pts_per_iter', y_col='true_value',
    comparison='across_sessions',session_indices=SESSION_SUBSETS['social'],min_pseudotrials=None
)
social_location_vs_pts_all_pts

Location during social trials, only sessions with at least 25 pseudotrials per training dataset

In [ ]:
if save_figs:
    save_path_social_location_enough_pts = os.path.join(pt_vs_decoding_path,'social_location_decoding_vs_num_pseudotrials_enough_pseudotrials.svg')
else:
    save_path_social_location_enough_pts = None

In [ ]:
fig, ax = plot_decoding_vs_pts(all_pop_data, null_options=NULL_OPTIONS, min_pseudotrials=min_pseudotrials_per_iter_social,
                               decode_type='location_social_across', metric='accuracy',
                               session_indices=SESSION_SUBSETS['social'], figsize=(6, 5), save_path=save_path_social_location_enough_pts)

In [ ]:
#stats
social_location_vs_pts_enough_pts = correlate_across_within_sessions(
    social_location_across_all_sessions, x_col='mean_train_pts_per_iter', y_col='true_value',
    comparison='across_sessions',session_indices=SESSION_SUBSETS['social'],min_pseudotrials=min_pseudotrials_per_iter_social
)
social_location_vs_pts_enough_pts

In [ ]:
social_location_vs_pts_all_pts['min_mean_pts_per_iter']=None
social_location_vs_pts_enough_pts['min_mean_pts_per_iter']=min_pseudotrials_per_iter_social

social_location_vs_pts_compare = pd.concat([social_location_vs_pts_all_pts,social_location_vs_pts_enough_pts])
social_location_vs_pts_compare

In [ ]:
save_path_social_location_decoding_vs_pts = os.path.join(pt_vs_decoding_path,'social_location_decoding_vs_num_training_pseudotrials.csv')
if save_csvs:
    social_location_vs_pts_compare.to_csv(save_path_social_location_decoding_vs_pts,index=None)

Location during the first empty trial, all sessions

In [ ]:
if save_figs:
    save_path_empty_location_all_pts = os.path.join(pt_vs_decoding_path,'empty_location_decoding_vs_num_pseudotrials_all_sessions.svg')
else:
    save_path_empty_location_all_pts = None

In [ ]:
fig, ax = plot_decoding_vs_pts(all_pop_data, null_options=NULL_OPTIONS, min_pseudotrials=None, decode_type='location_empty1_across', metric='accuracy',
                          session_indices=SESSION_SUBSETS['social'], figsize=(6, 5), save_path=save_path_empty_location_all_pts)

In [ ]:
#stats
empty_location_vs_pts_all_pts = correlate_across_within_sessions(
    empty_location_across_all_sessions, x_col='mean_train_pts_per_iter', y_col='true_value',
    comparison='across_sessions',session_indices=SESSION_SUBSETS['social'],min_pseudotrials=None
)
empty_location_vs_pts_all_pts

Location during empty trial 1, only sessions with at least 25 pseudotrials per training dataset

In [ ]:
if save_figs:
    save_path_empty_location_enough_pts = os.path.join(pt_vs_decoding_path,'empty_location_decoding_vs_num_pseudotrials_enough_pseudotrials.svg')
else:
    save_path_empty_location_enough_pts = None

In [ ]:
fig, ax = plot_decoding_vs_pts(all_pop_data, null_options=NULL_OPTIONS, min_pseudotrials=min_pseudotrials_per_iter_empty,
                               decode_type='location_empty1_across', metric='accuracy',
                               session_indices=SESSION_SUBSETS['social'], figsize=(6, 5), save_path=save_path_empty_location_enough_pts)

In [ ]:
#stats
empty_location_vs_pts_enough_pts = correlate_across_within_sessions(
    empty_location_across_all_sessions, x_col='mean_train_pts_per_iter', y_col='true_value',
    comparison='across_sessions',session_indices=SESSION_SUBSETS['social'],min_pseudotrials=min_pseudotrials_per_iter_empty
)
empty_location_vs_pts_enough_pts

In [ ]:
empty_location_vs_pts_all_pts['min_mean_pts_per_iter']=None
empty_location_vs_pts_enough_pts['min_mean_pts_per_iter']=min_pseudotrials_per_iter_empty

empty_location_vs_pts_compare = pd.concat([empty_location_vs_pts_all_pts,empty_location_vs_pts_enough_pts])
empty_location_vs_pts_compare

In [ ]:
save_path_empty_location_decoding_vs_pts = os.path.join(pt_vs_decoding_path,'empty_location_decoding_vs_num_training_pseudotrials.csv')
if save_csvs:
    empty_location_vs_pts_compare.to_csv(save_path_empty_location_decoding_vs_pts,index=None)

#### Decoding statistics

1) how many of a given metric's decoding runs (across animals and sessions) were significant relative to its own null? --> Benjamini-Hochberg on the P values across the three main decoding types -- location empty 1, location, identity (only ran null distributions for these three)


In [ ]:
all_decode_types_across_sessions['location_empty1']

In [ ]:
def correct_decoding_significance(df,alpha=0.05,min_pseudotrials=None):
    if min_pseudotrials is not None:
        results_df = df[df['mean_train_pts_per_iter']>=min_pseudotrials].copy()
    else:
        results_df = df.copy()
    valid_mask = results_df['p_value'].notna()
    num_valid = valid_mask.sum()
    if num_valid > 0:
        valid_pvals = results_df.loc[valid_mask, 'p_value'].values
        rejected, p_corrected, _, _ = multipletests(valid_pvals, alpha=alpha, method='fdr_bh')
        results_df['p_fdr'] = np.nan
        results_df.loc[valid_mask, 'p_fdr'] = p_corrected
        results_df['significant'] = False
        results_df.loc[valid_mask, 'significant'] = rejected

        num_sig = results_df['significant'].sum()
        prop_sig = num_sig/num_valid
        print(f'% of significant decoding runs for {results_df["decode_type"].iloc[0]} across all animals and sessions: {prop_sig}')
        
    else:
        results_df['p_fdr'] = np.nan
        results_df['significant'] = False
        prop_sig = np.nan
        print('No valid decoding runs; cannot assess significance')
    return(prop_sig, results_df)

In [ ]:
_, location_empty1_decoding_all_sessions = correct_decoding_significance(all_decode_types_across_sessions['location_empty1'],alpha=0.05,min_pseudotrials=None)
_, location_empty1_decoding_enough_pts = correct_decoding_significance(all_decode_types_across_sessions['location_empty1'],alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_empty)
location_empty1_decoding_enough_pts

In [ ]:
if save_csvs:
    location_empty1_decoding_all_sessions.to_csv(os.path.join(decoding_path,'empty_trial_location_decoding_results_all_sessions_corrected.csv'),
                                                index=False)
    location_empty1_decoding_enough_pts.to_csv(os.path.join(decoding_path,'empty_trial_location_decoding_results_enough_pts_sessions_corrected.csv'),
                                                index=False)

In [ ]:
_, location_social_decoding_all_sessions = correct_decoding_significance(all_decode_types_across_sessions['location_social'],alpha=0.05,min_pseudotrials=None)
_, location_social_decoding_enough_pts = correct_decoding_significance(all_decode_types_across_sessions['location_social'],alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_social)
location_social_decoding_enough_pts

In [ ]:
if save_csvs:
    location_social_decoding_all_sessions.to_csv(os.path.join(decoding_path,'social_trials_location_decoding_results_all_sessions_corrected.csv'),
                                                index=False)
    location_social_decoding_enough_pts.to_csv(os.path.join(decoding_path,'social_trials_location_decoding_results_enough_pts_sessions_corrected.csv'),
                                                index=False)

In [ ]:
_, identity_decoding_all_sessions = correct_decoding_significance(all_decode_types_across_sessions['identity'],alpha=0.05,min_pseudotrials=None)
_, identity_decoding_enough_pts = correct_decoding_significance(all_decode_types_across_sessions['identity'],alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_social)
identity_decoding_enough_pts

In [ ]:
if save_csvs:
    identity_decoding_all_sessions.to_csv(os.path.join(decoding_path,'identity_decoding_results_all_sessions_corrected.csv'),
                                                index=False)
    identity_decoding_enough_pts.to_csv(os.path.join(decoding_path,'identity_decoding_results_enough_pts_sessions_corrected.csv'),
                                                index=False)

2) across animals, is decoding of each variable significant against null at each time point?
-->wilcoxon signed-rank between true performance and mean null performance at each timepoint + Benjamini-Hochberg correction for multiple sessions

In [ ]:
identity_decoding_sig_by_timepoint_all_sessions = test_decoding_group_significance(all_pop_data,decoding_types=DECODING_TYPES,decode_type='identity',metric='accuracy',
                                 session_indices=SESSION_SUBSETS['social'],verbose=False,alpha=0.05,min_pseudotrials=None)
identity_decoding_sig_by_timepoint_all_sessions

In [ ]:
identity_decoding_sig_by_timepoint_enough_pts = test_decoding_group_significance(all_pop_data,decoding_types=DECODING_TYPES,decode_type='identity',metric='accuracy',
                                 session_indices=SESSION_SUBSETS['social'],verbose=False,alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_social)
identity_decoding_sig_by_timepoint_enough_pts

In [ ]:
if save_csvs:
    identity_decoding_sig_by_timepoint_all_sessions.to_csv(os.path.join(within_sesh_path,'identity_decoding_wilcox_by_timepoint_all_sessions_corrected.csv'),
                                                index=False)
    identity_decoding_sig_by_timepoint_enough_pts.to_csv(os.path.join(within_sesh_path,'identity_decoding_wilcox_by_timepoint_enough_pts_sessions_corrected.csv'),
                                                index=False)

In [ ]:
within_sesh_path

In [ ]:
social_location_decoding_sig_by_timepoint_all_sessions = test_decoding_group_significance(all_pop_data,decoding_types=DECODING_TYPES,decode_type='location_social',metric='accuracy',
                                 session_indices=SESSION_SUBSETS['social'],verbose=False,alpha=0.05,min_pseudotrials=None)
social_location_decoding_sig_by_timepoint_all_sessions

In [ ]:
social_location_decoding_sig_by_timepoint_enough_pts = test_decoding_group_significance(all_pop_data,decoding_types=DECODING_TYPES,decode_type='location_social',metric='accuracy',
                                 session_indices=SESSION_SUBSETS['social'],verbose=False,alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_social)
social_location_decoding_sig_by_timepoint_enough_pts

In [ ]:
if save_csvs:
    social_location_decoding_sig_by_timepoint_all_sessions.to_csv(os.path.join(within_sesh_path,'social_location_decoding_wilcox_by_timepoint_all_sessions_corrected.csv'),
                                                index=False)
    social_location_decoding_sig_by_timepoint_enough_pts.to_csv(os.path.join(within_sesh_path,'social_location_decoding_wilcox_by_timepoint_enough_pts_sessions_corrected.csv'),
                                                index=False)

In [ ]:
empty_location_decoding_sig_by_timepoint_all_sessions = test_decoding_group_significance(all_pop_data,decoding_types=DECODING_TYPES,decode_type='location_empty1',metric='accuracy',
                                 session_indices=SESSION_SUBSETS['social'],verbose=False,alpha=0.05,min_pseudotrials=None)
empty_location_decoding_sig_by_timepoint_all_sessions

In [ ]:
empty_location_decoding_sig_by_timepoint_enough_pts = test_decoding_group_significance(all_pop_data,decoding_types=DECODING_TYPES,decode_type='location_empty1',metric='accuracy',
                                 session_indices=SESSION_SUBSETS['social'],verbose=False,alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_empty)
empty_location_decoding_sig_by_timepoint_enough_pts

In [ ]:
if save_csvs:
    empty_location_decoding_sig_by_timepoint_all_sessions.to_csv(os.path.join(within_sesh_path,'empty_location_decoding_wilcox_by_timepoint_all_sessions_corrected.csv'),
                                                index=False)
    empty_location_decoding_sig_by_timepoint_enough_pts.to_csv(os.path.join(within_sesh_path,'empty_location_decoding_wilcox_by_timepoint_enough_pts_sessions_corrected.csv'),
                                                index=False)

3) Does decoding performance significantly differ within animals across sessions? --> Wilcoxon signed-rank test across multiple session comparisons with Benjamini-Hochberg correction for # of comparisons

Note that the # of neurons is held constant across all sessions within an animal (randomly downsampled to the minimum # for any given session)

In [ ]:
#decide whether to drop animals with fewer than 25 pseudotrials from this comparison, but probably don't -- we need as many samples as we can get, here

In [ ]:
comparisons_of_interest = ['same_sex','os_presep','novel','familiar_presep','os_familiar']

In [ ]:
def decoding_comparison_loop(comparisons_of_interest,all_pop_data,decode_type,null_options,decoding_types,
                             metric='accuracy',test='wilcoxon',alternative='two-sided',alpha=0.05,min_pseudotrials=None,verbose=False):
    res_list = []
    for comp in comparisons_of_interest:
        sesh_idx = SESSION_SUBSETS[comp]
        res = compare_decoding_sessions(all_pop_data,decode_type=decode_type,session_indices=sesh_idx,null_options=null_options,decoding_types=decoding_types,
            metric=metric,test=test,alternative=alternative,min_pseudotrials=min_pseudotrials,verbose=verbose)
        if res is not None:
            res_list.append(res)
    all_comps_df = pd.DataFrame(res_list)

    valid_mask = all_comps_df['p_value'].notna()
    if valid_mask.sum() > 0:
        pvals = all_comps_df.loc[valid_mask,'p_value'].values
        rejected, p_corrected, _, _ = multipletests(pvals,alpha=alpha,method='fdr_bh')
        all_comps_df['p_fdr']=np.nan
        all_comps_df.loc[valid_mask,'p_fdr']=p_corrected
        all_comps_df['significant']=False
        all_comps_df.loc[valid_mask,'significant']=rejected
    all_comps_df = all_comps_df.rename(columns={'p_value':'p_uncorrected'})
    
    return(all_comps_df)

Identity decoding

In [ ]:
identity_decoding_across_timepoints_sig_all_sessions = decoding_comparison_loop(comparisons_of_interest,all_pop_data,decode_type='identity_within',
                                                                                null_options=NULL_OPTIONS,decoding_types=DECODING_TYPES,
                                                                                metric='accuracy',test='wilcoxon',alternative='two-sided',
                                                                                alpha=0.05,min_pseudotrials=None,verbose=False)
identity_decoding_across_timepoints_sig_all_sessions

In [ ]:
identity_decoding_across_timepoints_sig_enough_pts = decoding_comparison_loop(comparisons_of_interest,all_pop_data,decode_type='identity_within',
                                                                              null_options=NULL_OPTIONS,decoding_types=DECODING_TYPES,
                                                                              metric='accuracy',test='wilcoxon',alternative='two-sided',
                                                                              alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_social,verbose=False)
identity_decoding_across_timepoints_sig_enough_pts

In [ ]:
if save_csvs:
    identity_decoding_across_timepoints_sig_all_sessions.to_csv(os.path.join(across_sesh_path,'identity_decoding_wilcox_across_timepoints_all_sessions_corrected.csv'),
                                                index=False)
    identity_decoding_across_timepoints_sig_enough_pts.to_csv(os.path.join(across_sesh_path,'identity_decoding_wilcox_across_timepoints_enough_pts_sessions_corrected.csv'),
                                                index=False)

Location decoding during the social trials

In [ ]:
social_location_decoding_across_timepoints_sig_all_sessions = decoding_comparison_loop(comparisons_of_interest,all_pop_data,decode_type='location_social_within',
                                                                                null_options=NULL_OPTIONS,decoding_types=DECODING_TYPES,
                                                                                metric='accuracy',test='wilcoxon',alternative='two-sided',
                                                                                alpha=0.05,min_pseudotrials=None,verbose=False)
social_location_decoding_across_timepoints_sig_all_sessions

In [ ]:
social_location_decoding_across_timepoints_sig_enough_pts = decoding_comparison_loop(comparisons_of_interest,all_pop_data,decode_type='location_social_within',
                                                                                null_options=NULL_OPTIONS,decoding_types=DECODING_TYPES,
                                                                                metric='accuracy',test='wilcoxon',alternative='two-sided',
                                                                                alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_social,verbose=False)
social_location_decoding_across_timepoints_sig_enough_pts

In [ ]:
if save_csvs:
    social_location_decoding_across_timepoints_sig_all_sessions.to_csv(os.path.join(across_sesh_path,'social_location_decoding_wilcox_across_timepoints_all_sessions_corrected.csv'),
                                                index=False)
    social_location_decoding_across_timepoints_sig_enough_pts.to_csv(os.path.join(across_sesh_path,'social_location_decoding_wilcox_across_timepoints_enough_pts_sessions_corrected.csv'),
                                                index=False)

Location decoding across the first empty trial

In [ ]:
empty_location_decoding_across_timepoints_sig_all_sessions = decoding_comparison_loop(comparisons_of_interest,all_pop_data,decode_type='location_empty1_within',
                                                                                null_options=NULL_OPTIONS,decoding_types=DECODING_TYPES,
                                                                                metric='accuracy',test='wilcoxon',alternative='two-sided',
                                                                                alpha=0.05,min_pseudotrials=None,verbose=False)
empty_location_decoding_across_timepoints_sig_all_sessions

In [ ]:
empty_location_decoding_across_timepoints_sig_enough_pts = decoding_comparison_loop(comparisons_of_interest,all_pop_data,decode_type='location_empty1_within',
                                                                                null_options=NULL_OPTIONS,decoding_types=DECODING_TYPES,
                                                                                metric='accuracy',test='wilcoxon',alternative='two-sided',
                                                                                alpha=0.05,min_pseudotrials=min_pseudotrials_per_iter_empty,verbose=False)
empty_location_decoding_across_timepoints_sig_enough_pts

In [ ]:
if save_csvs:
    empty_location_decoding_across_timepoints_sig_all_sessions.to_csv(os.path.join(across_sesh_path,'empty_location_decoding_wilcox_across_timepoints_all_sessions_corrected.csv'),
                                                index=False)
    empty_location_decoding_across_timepoints_sig_enough_pts.to_csv(os.path.join(across_sesh_path,'empty_location_decoding_wilcox_across_timepoints_enough_pts_sessions_corrected.csv'),
                                                index=False)

4. Does decoding performance signficantly differ within animals within sessions across different variables? Specifically, is identity decoding worse than location decoding during the social trials, and is location decoding significantly different across the first empty cup trial and the social trials?
   --> wilcoxon within session bw each animal's decoding performance metric on two different variables for each session, with Benjamini-Hochberg correction for multiple comparisons

In [ ]:
#define the dfs as needed for the wilcoxon within-session function
id_decode_across_sessions_all_sessions = all_decode_types_across_sessions['identity'].copy().rename(columns={'true_value':'value'})
id_decode_across_sessions_enough_pts = id_decode_across_sessions_all_sessions[id_decode_across_sessions_all_sessions['mean_train_pts_per_iter']>=min_pseudotrials_per_iter_social]

social_location_decode_across_sessions_all_sessions = all_decode_types_across_sessions['location_social'].copy().rename(columns={'true_value':'value'})
social_location_decode_across_sessions_enough_pts = social_location_decode_across_sessions_all_sessions[social_location_decode_across_sessions_all_sessions['mean_train_pts_per_iter']>=min_pseudotrials_per_iter_social]

empty_location_decode_across_sessions_all_sessions = all_decode_types_across_sessions['location_empty1'].copy().rename(columns={'true_value':'value'})
empty_location_decode_across_sessions_enough_pts = empty_location_decode_across_sessions_all_sessions[empty_location_decode_across_sessions_all_sessions['mean_train_pts_per_iter']>=min_pseudotrials_per_iter_empty]

In [ ]:
id_decode_across_sessions_all_sessions

In [ ]:
id_vs_location_across_timepoints_all_sessions = compare_metrics_within_sessions(id_decode_across_sessions_all_sessions,social_location_decode_across_sessions_all_sessions,label_a='identity',label_b='social_location',
                               session_indices=None,test='wilcoxon',alternative='two-sided',addl_info=None,alpha=0.05)

id_vs_location_across_timepoints_enough_pts = compare_metrics_within_sessions(id_decode_across_sessions_enough_pts,social_location_decode_across_sessions_enough_pts,label_a='identity',label_b='social_location',
                               session_indices=None,test='wilcoxon',alternative='two-sided',addl_info=None,alpha=0.05)

id_vs_location_across_timepoints_enough_pts

In [ ]:
if save_csvs:
    id_vs_location_across_timepoints_all_sessions.to_csv(os.path.join(within_sesh_path,'id_vs_location_decoding_across_timepoints_sig_all_sessions.csv'),
                                                         index=False)
    id_vs_location_across_timepoints_enough_pts.to_csv(os.path.join(within_sesh_path,'id_vs_location_decoding_across_timepoints_sig_enough_pts.csv'),
                                                         index=False)

In [ ]:
loc_vs_loc_across_timepoints_all_sessions = compare_metrics_within_sessions(empty_location_decode_across_sessions_all_sessions,social_location_decode_across_sessions_all_sessions,label_a='empty_location',label_b='social_location',
                               session_indices=None,test='wilcoxon',alternative='two-sided',addl_info=None,alpha=0.05)

loc_vs_loc_across_timepoints_enough_pts = compare_metrics_within_sessions(empty_location_decode_across_sessions_enough_pts,social_location_decode_across_sessions_enough_pts,label_a='empty_location',label_b='social_location',
                               session_indices=None,test='wilcoxon',alternative='two-sided',addl_info=None,alpha=0.05)

loc_vs_loc_across_timepoints_enough_pts

In [ ]:
if save_csvs:
    loc_vs_loc_across_timepoints_all_sessions.to_csv(os.path.join(within_sesh_path,'empty_location_vs_social_location_decoding_across_timepoints_sig_all_sessions.csv'),
                                                         index=False)
    loc_vs_loc_across_timepoints_enough_pts.to_csv(os.path.join(within_sesh_path,'empty_location_vs_social_location_decoding_across_timepoints_sig_enough_pts.csv'),
                                                         index=False)

#### Decoding plotting

Define plotting functions

In [ ]:
def plot_decoding_single_session(all_data, session_idx, metric='accuracy',
                                  decode_types=None, null_type='circular_aware',
                                  show_individual_animals=True,
                                  figsize=(8, 6), save_path=None,
                                  show=True):
    """
    Plot decoding performance for multiple decode types at a single session,
    showing true values vs null distributions.
    
    Parameters:
    -----------
    session_idx : int
        Which session to plot
    metric : str
        'accuracy', 'f1', or 'auroc'
    decode_types : list or None
        Which decode types to include. If None, uses all.
        Options: 'location_empty1', 'location_social', 'identity'
    null_type : str
        'circular', 'circular_aware', or 'shuffle'
    show_individual_animals : bool
        Whether to show individual animal points
    """
    if decode_types is None:
        decode_types = list(DECODING_TYPES.keys())
    
    # Extract data for all decode types
    df = extract_all_decoding_types(all_data, decoding_types=DECODING_TYPES, null_options=NULL_OPTIONS, metric=metric, session_idx=session_idx,
                                     null_type=null_type)
    df = df[df['decode_type'].isin(decode_types)]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    x_positions = np.arange(len(decode_types))
    # Tighter spacing when only 1 decode type
    offset = 0.15 if len(decode_types) > 1 else 0.08
    x_labels = [DECODING_TYPES[dt]['short_label'] for dt in decode_types]
    colors = [DECODING_TYPES[dt]['color'] for dt in decode_types]
    
    # Plot null distributions (individual animal means + group mean/SEM)
    for i, dt in enumerate(decode_types):
        dt_data = df[df['decode_type'] == dt]
        
        # Compute per-animal null means
        animal_null_means = []
        for _, row in dt_data.iterrows():
            null_vals = row['null_values']
            if isinstance(null_vals, (list, np.ndarray)) and len(null_vals) > 0:
                animal_null_means.append(np.mean(null_vals))
        
        if len(animal_null_means) > 0:
            animal_null_means = np.array(animal_null_means)
            
            # Individual animal null means
            if show_individual_animals:
                ax.scatter([i + offset] * len(animal_null_means), animal_null_means,
                          c='gray', s=60, alpha=0.4, edgecolors='white', linewidths=0.5)
            
            # Group mean ± SEM
            group_null_mean = np.mean(animal_null_means)
            group_null_sem = (np.std(animal_null_means) / np.sqrt(len(animal_null_means))
                             if len(animal_null_means) > 1 else 0)
            
            ax.errorbar(i + offset, group_null_mean, yerr=group_null_sem,
                       fmt='o', color='gray', markersize=12,
                       markeredgecolor='black', markeredgewidth=1.5,
                       capsize=5, capthick=2, linewidth=2,
                       label='Null' if i == 0 else None, alpha=0.7)
    
    # Plot true values
    for i, dt in enumerate(decode_types):
        dt_data = df[df['decode_type'] == dt].dropna(subset=['true_value'])
        color = DECODING_TYPES[dt]['color']
        
        if show_individual_animals:
            # Individual animal points
            ax.scatter([i - offset] * len(dt_data), dt_data['true_value'], 
                      c=color, s=60, alpha=0.6, edgecolors='white', linewidths=0.5)
        
        # Mean with error bar
        mean_val = dt_data['true_value'].mean()
        sem_val = dt_data['true_value'].sem() if len(dt_data) > 1 else 0
        
        ax.errorbar(i - offset, mean_val, yerr=sem_val,
                   fmt='s', color=color, markersize=12,
                   markeredgecolor='black', markeredgewidth=1.5,
                   capsize=5, capthick=2, linewidth=2)
    
    # Chance line
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    
    # Formatting
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, fontsize=11)

    if len(decode_types) == 1:
        ax.set_xlim(-0.5, 0.5)
    
    metric_labels = {'accuracy': 'Accuracy', 'f1': 'F1 Score', 'auroc': 'AUROC'}
    ax.set_ylabel(metric_labels.get(metric, metric), fontsize=12)
    
    session_label = SESSION_CONFIG['short_labels'][session_idx]
    ax.set_title(f'Decoding Performance - {session_label}', fontsize=14)
    
    ax.set_ylim(0.45, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Legend
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', markeredgecolor='black', markerfacecolor='white', 
               markersize=10, label='True mean'),
        Line2D([0], [0], marker='o', color='w', markeredgecolor='black', markerfacecolor='gray',
               markersize=10, label='Null mean', alpha=0.7),
    ]
    ax.legend(handles=legend_elements, loc='lower right')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    
    if show:  # Only show if requested
        plt.show()
    return fig, ax

def plot_decoding_single_session_paired(all_data, session_idx, metric='accuracy',
                                         decode_types=None, null_type='circular_aware',
                                         min_pseudotrials=None, min_pseudotrials_empty=None,
                                         show_individual_animals=True, show_means=True,
                                         figsize=(8, 6), save_path=None, show=True):
    """
    Plot decoding performance vs null for one session, with paired lines
    connecting each animal's true value to their null mean.
    """
    if decode_types is None:
        decode_types = list(DECODING_TYPES.keys())

    # If no separate empty threshold provided, use the social one
    if min_pseudotrials_empty is None:
        min_pseudotrials_empty = min_pseudotrials

    # Decode types that use empty-trial data
    EMPTY_DECODE_TYPES = {'location_empty1', 'location_empty1_within', 'location_empty1_across'}
    
    df = extract_all_decoding_types(all_data, decoding_types=DECODING_TYPES,
                                     null_options=NULL_OPTIONS, metric=metric,
                                     session_idx=session_idx, null_type=null_type)
    df = df[df['decode_type'].isin(decode_types)]
    
    fig, ax = plt.subplots(figsize=figsize)

    # Each decode type gets two x-positions: left = true, right = null
    spacing = 0.8       # distance between decode type groups
    pair_width = 0.3    # distance between true and null within a pair

    for i, dt in enumerate(decode_types):
        dt_data = df[df['decode_type'] == dt].dropna(subset=['true_value'])

        threshold = min_pseudotrials_empty if dt in EMPTY_DECODE_TYPES else min_pseudotrials
        if threshold is not None:
            dt_data = dt_data[dt_data['mean_train_pts_per_iter'] >= threshold]
        
        color = DECODING_TYPES[dt]['color']

        x_true = i * spacing
        x_null = i * spacing + pair_width

        # Collect per-animal null means
        animal_true = []
        animal_null = []
        for _, row in dt_data.iterrows():
            null_vals = row['null_values']
            if isinstance(null_vals, (list, np.ndarray)) and len(null_vals) > 0:
                animal_true.append(row['true_value'])
                animal_null.append(np.mean(null_vals))

        animal_true = np.array(animal_true)
        animal_null = np.array(animal_null)

        if show_individual_animals and len(animal_true) > 0:
            # Paired lines connecting each animal's true to null
            for t, n in zip(animal_true, animal_null):
                ax.plot([x_true, x_null], [t, n], '-', color='gray',
                        alpha=0.65, linewidth=1)

            # Individual points
            ax.scatter([x_true] * len(animal_true), animal_true,
                       c=color, s=60, alpha=0.6, edgecolors='white',
                       linewidths=0.5, zorder=5)
            ax.scatter([x_null] * len(animal_null), animal_null,
                       c='gray', s=60, alpha=0.4, edgecolors='white',
                       linewidths=0.5, zorder=5)
        
        # Group means ± SEM
        if show_means:
            if len(animal_true) > 0:
                true_mean = np.mean(animal_true)
                true_sem = np.std(animal_true) / np.sqrt(len(animal_true)) if len(animal_true) > 1 else 0
                ax.errorbar(x_true, true_mean, yerr=true_sem,
                            fmt='s', color=color, markersize=12,
                            markeredgecolor='black', markeredgewidth=1.5,
                            capsize=5, capthick=2, linewidth=2, zorder=10)

            if len(animal_null) > 0:
                null_mean = np.mean(animal_null)
                null_sem = np.std(animal_null) / np.sqrt(len(animal_null)) if len(animal_null) > 1 else 0
                ax.errorbar(x_null, null_mean, yerr=null_sem,
                            fmt='o', color='gray', markersize=12,
                            markeredgecolor='black', markeredgewidth=1.5,
                            capsize=5, capthick=2, linewidth=2, alpha=0.7, zorder=10)

    # Chance line
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, linewidth=1)

    # X-axis: label each decode type centered between its true/null positions
    x_centers = [i * spacing + pair_width / 2 for i in range(len(decode_types))]
    x_labels = [DECODING_TYPES[dt]['label'] for dt in decode_types]
    ax.set_xticks(x_centers)
    ax.set_xticklabels(x_labels, fontsize=11)

    # Pad x-axis
    ax.set_xlim(-0.3, (len(decode_types) - 1) * spacing + pair_width + 0.3)

    metric_labels = {'accuracy': 'Accuracy', 'f1': 'F1 Score', 'auroc': 'AUROC'}
    ax.set_ylabel(metric_labels.get(metric, metric), fontsize=12)

    session_label = SESSION_CONFIG['short_labels'][session_idx]
    ax.set_title(f'Decoding Performance — {session_label}', fontsize=14)

    ax.set_ylim(0.45, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legend
    from matplotlib.lines import Line2D
    if show_means:
        legend_elements = [
            Line2D([0], [0], marker='s', color='w', markeredgecolor='black',
                   markerfacecolor='white', markersize=10, label='True mean'),
            Line2D([0], [0], marker='o', color='w', markeredgecolor='black',
                   markerfacecolor='gray', markersize=10, label='Null mean', alpha=0.7),
        ]
        ax.legend(handles=legend_elements, loc='upper right')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)

    if show:
        plt.show()
    return fig, ax


Plot decoding performance against null within a session

In [ ]:
within_sesh_figsize=(6,5)

In [ ]:
within_sesh_figpath=os.path.join(within_sesh_path,'figures')

same-sex novel/novel (session 1)

In [ ]:
if save_figs:
    ssN_fig_path_all_sessions=os.path.join(within_sesh_figpath,'all_decode_types_ss_novel_all_sessions.svg')
    ssN_fig_path_enough_pts=os.path.join(within_sesh_figpath,'all_decode_types_ss_novel_enough_pt_sessions.svg')
else:
    ssN_fig_path_all_sessions=None
    ssN_fig_path_enough_pts=None

In [ ]:
fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=1, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=None,min_pseudotrials_empty=None,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=ssN_fig_path_all_sessions, show=True)

fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=1, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=min_pseudotrials_per_iter_social,min_pseudotrials_empty=min_pseudotrials_per_iter_empty,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=ssN_fig_path_enough_pts, show=True)

same-sex novel/familiar (session 2)

In [ ]:
if save_figs:
    ssF_fig_path_all_sessions=os.path.join(within_sesh_figpath,'all_decode_types_ss_familiar_all_sessions.svg')
    ssF_fig_path_enough_pts=os.path.join(within_sesh_figpath,'all_decode_types_ss_familiar_enough_pt_sessions.svg')
else:
    ssF_fig_path_all_sessions=None
    ssF_fig_path_enough_pts=None

In [ ]:
fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=2, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=None,min_pseudotrials_empty=None,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=ssF_fig_path_all_sessions, show=True)

fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=2, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=min_pseudotrials_per_iter_social,min_pseudotrials_empty=min_pseudotrials_per_iter_empty,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=ssF_fig_path_enough_pts, show=True)

opposite-sex novel/novel (session 3)

In [ ]:
if save_figs:
    osN_fig_path_all_sessions=os.path.join(within_sesh_figpath,'all_decode_types_os_novel_all_sessions.svg')
    osN_fig_path_enough_pts=os.path.join(within_sesh_figpath,'all_decode_types_os_novel_enough_pt_sessions.svg')
else:
    osN_fig_path_all_sessions=None
    osN_fig_path_enough_pts=None

In [ ]:
fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=3, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=None,min_pseudotrials_empty=None,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=osN_fig_path_all_sessions, show=True)

fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=3, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=min_pseudotrials_per_iter_social,min_pseudotrials_empty=min_pseudotrials_per_iter_empty,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=osN_fig_path_enough_pts, show=True)

opposite-sex novel/familiar pre-separation (session 4)

In [ ]:
if save_figs:
    osF_presep_fig_path_all_sessions=os.path.join(within_sesh_figpath,'all_decode_types_os_familiar_presep_all_sessions.svg')
    osF_presep_fig_path_enough_pts=os.path.join(within_sesh_figpath,'all_decode_types_os_familiar_presep_enough_pt_sessions.svg')
else:
    osF_presep_fig_path_all_sessions=None
    osF_presep_fig_path_enough_pts=None

In [ ]:
fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=4, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=None,min_pseudotrials_empty=None,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=osF_presep_fig_path_all_sessions, show=True)

fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=4, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=min_pseudotrials_per_iter_social,min_pseudotrials_empty=min_pseudotrials_per_iter_empty,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=osF_presep_fig_path_enough_pts, show=True)

opposite-sex novel/familiar post-separation (session 5)

In [ ]:
if save_figs:
    osF_postsep_fig_path_all_sessions=os.path.join(within_sesh_figpath,'all_decode_types_os_familiar_postsep_all_sessions.svg')
    osF_postsep_fig_path_enough_pts=os.path.join(within_sesh_figpath,'all_decode_types_os_familiar_postsep_enough_pt_sessions.svg')
else:
    osF_postsep_fig_path_all_sessions=None
    osF_postsep_fig_path_enough_pts=None

In [ ]:
fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=5, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=None,min_pseudotrials_empty=None,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=osF_postsep_fig_path_all_sessions, show=True)

fig, ax = plot_decoding_single_session_paired(all_pop_data, session_idx=5, metric='accuracy',
                                         decode_types=['location_empty1','location_social','identity'], null_type='circular_aware',
                                         min_pseudotrials=min_pseudotrials_per_iter_social,min_pseudotrials_empty=min_pseudotrials_per_iter_empty,
                                         show_individual_animals=True,show_means=True,
                                         figsize=within_sesh_figsize, save_path=osF_postsep_fig_path_enough_pts, show=True)

Plot decoding performance across sessions

In [ ]:
across_sesh_figpath = os.path.join(across_sesh_path,'figures')

In [ ]:
def plot_decoding_trajectory(all_data, decode_type='identity_within', metric='accuracy',
                              session_indices=None, null_type='circular_aware',
                              show_null=False, min_pseudotrials=None, 
                              title_specifier=None, figsize=(10, 6), save_path=None):
    """
    Plot decoding performance across sessions with individual animal lines.
    
    Parameters:
    -----------
    decode_type : str
        Which decoding type to plot
    metric : str
        'accuracy', 'f1', or 'auroc'
    session_indices : list or None
        Which sessions to include
    show_null : bool
        Whether to show null distribution as shaded region
    """
    if session_indices is None:
        session_indices = SESSION_SUBSETS['social']
    
    df = extract_decoding_metric(all_data, decode_type, null_options=NULL_OPTIONS, metric=metric,
                                  session_indices=session_indices, animals=None, include_null=True,
                                  null_type=null_type)
    if min_pseudotrials is not None:
        df = df[df['mean_train_pts_per_iter']>=min_pseudotrials]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    x_positions = list(range(len(session_indices)))
    x_labels = [SESSION_CONFIG['short_labels'][i] for i in session_indices]
    colors = [SESSION_CONFIG['colors'][i] for i in session_indices]
    decode_color = DECODING_TYPES[decode_type]['color']
    
    # Plot null distribution as shaded region
    if show_null:
        null_means = []
        null_stds = []
        for s_idx in session_indices:
            s_data = df[df['session_idx'] == s_idx]
            # Pool all null values
            all_null = []
            for nv in s_data['null_values']:
                if isinstance(nv, (list, np.ndarray)):
                    all_null.extend(nv)
            null_means.append(np.mean(all_null) if all_null else np.nan)
            null_stds.append(np.std(all_null) if all_null else 0)
        
        null_means = np.array(null_means)
        null_stds = np.array(null_stds)
        
        ax.fill_between(x_positions, null_means - null_stds, null_means + null_stds,
                        color='gray', alpha=0.2, label='Null ± SD')
        ax.plot(x_positions, null_means, 'o--', color='gray', alpha=0.5, markersize=6)
    
    # Plot individual animal trajectories
    for animal in df['animal'].unique():
        animal_data = df[df['animal'] == animal].set_index('session_idx')
        y_values = [animal_data.loc[s, 'true_value'] if s in animal_data.index else np.nan 
                    for s in session_indices]
        ax.plot(x_positions, y_values, 'o-', color=decode_color, alpha=0.4, 
                markersize=8, linewidth=1.5, label=animal)
    
    # Plot group mean with error bars
    means = []
    sems = []
    for s_idx in session_indices:
        s_data = df[df['session_idx'] == s_idx]['true_value'].dropna()
        means.append(s_data.mean())
        sems.append(s_data.sem() if len(s_data) > 1 else 0)
    
    ax.errorbar(x_positions, means, yerr=sems, fmt='s-', color='black',
                markersize=12, linewidth=2.5, capsize=5, capthick=2, zorder=10)
    
    # Chance line
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, linewidth=1)
    
    # Color x-axis labels by session
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, fontsize=11)
    for ticklabel, color in zip(ax.get_xticklabels(), colors):
        ticklabel.set_color(color)
        ticklabel.set_fontweight('bold')
    
    metric_labels = {'accuracy': 'Accuracy', 'f1': 'F1 Score', 'auroc': 'AUROC'}
    ax.set_ylabel(metric_labels.get(metric, metric), fontsize=12)
    ax.set_xlabel('Timepoint', fontsize=12)
    
    decode_label = DECODING_TYPES[decode_type]['label']
    if min_pseudotrials is not None:
        pt_disclaimer=' (sessions meeting min. training data threshold)'
    else:
        pt_disclaimer=', all sessions'
        
    if title_specifier:
        ax.set_title(f'{decode_label} decoding across {title_specifier} timepoints{pt_disclaimer}', fontsize=14)
    else:
        ax.set_title(f'{decode_label} decoding across timepoints{pt_disclaimer}', fontsize=14)
    
    ax.set_ylim(0.45, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    return fig, ax

In [ ]:
across_sesh_figsize = (8,4)

Identity decoding, all sessions

In [ ]:
if save_figs:
    identity_all_sessions_fig_path=os.path.join(across_sesh_figpath,'identity_decoding_all_sessions.svg')
    identity_all_sessions_enough_pts_fig_path=os.path.join(across_sesh_figpath,'identity_decoding_all_sessions_enough_pts.svg')
else:
    identity_all_sessions_fig_path=None
    identity_all_sessions_enough_pts_fig_path=None

In [ ]:
fig, ax = plot_decoding_trajectory(all_pop_data, decode_type='identity_within', metric='accuracy',
                              session_indices=SESSION_SUBSETS['social'], null_type='circular_aware',
                              show_null=False, min_pseudotrials=None,title_specifier=None,figsize=across_sesh_figsize,
                                   save_path=identity_all_sessions_fig_path)

In [ ]:
fig, ax = plot_decoding_trajectory(all_pop_data, decode_type='identity_within', metric='accuracy',
                              session_indices=SESSION_SUBSETS['social'], null_type='circular_aware',
                              show_null=False, min_pseudotrials=min_pseudotrials_per_iter_social,title_specifier=None,
                                   figsize=across_sesh_figsize, save_path=identity_all_sessions_enough_pts_fig_path)

Location during social trials, all sessions

In [ ]:
if save_figs:
    social_location_all_sessions_fig_path=os.path.join(across_sesh_figpath,'social_location_decoding_all_sessions.svg')
    social_location_all_sessions_enough_pts_fig_path=os.path.join(across_sesh_figpath,'social_location_decoding_all_sessions_enough_pts.svg')
else:
    social_location_all_sessions_fig_path=None
    social_location_all_sessions_enough_pts_fig_path=None

In [ ]:
fig, ax = plot_decoding_trajectory(all_pop_data, decode_type='location_social_within', metric='accuracy',
                              session_indices=SESSION_SUBSETS['social'], null_type='circular_aware',
                              show_null=False, min_pseudotrials=None,title_specifier=None,
                                   figsize=across_sesh_figsize, save_path=social_location_all_sessions_fig_path)

In [ ]:
fig, ax = plot_decoding_trajectory(all_pop_data, decode_type='location_social_within', metric='accuracy',
                              session_indices=SESSION_SUBSETS['social'], null_type='circular_aware',
                              show_null=False, min_pseudotrials=min_pseudotrials_per_iter_social,title_specifier=None,
                                   figsize=across_sesh_figsize, save_path=social_location_all_sessions_enough_pts_fig_path)

Location during first empty trial, all sessions

In [ ]:
if save_figs:
    empty_location_all_sessions_fig_path=os.path.join(across_sesh_figpath,'empty_location_decoding_all_sessions.svg')
    empty_location_all_sessions_enough_pts_fig_path=os.path.join(across_sesh_figpath,'empty_location_decoding_all_sessions_enough_pts.svg')
else:
    empty_location_all_sessions_fig_path=None
    empty_location_all_sessions_enough_pts_fig_path=None

In [ ]:
fig, ax = plot_decoding_trajectory(all_pop_data, decode_type='location_empty1_within', metric='accuracy',
                              session_indices=SESSION_SUBSETS['social'], null_type='circular_aware',
                              show_null=False, min_pseudotrials=None,title_specifier=None,
                                   figsize=across_sesh_figsize,save_path=empty_location_all_sessions_fig_path)

In [ ]:
fig, ax = plot_decoding_trajectory(all_pop_data, decode_type='location_empty1_within', metric='accuracy',
                              session_indices=SESSION_SUBSETS['social'], null_type='circular_aware',
                              show_null=False, min_pseudotrials=min_pseudotrials_per_iter_empty,title_specifier=None,
                                   figsize=across_sesh_figsize,save_path=empty_location_all_sessions_enough_pts_fig_path)

Loop to print figures for every comparison, each decoding type, and +/- minimum pseudotrials per training dataset and save out

In [ ]:
comparisons_of_interest

In [ ]:
within_decode_types = ['location_empty1_within','location_social_within','identity_within']
min_pt_options = [min_pseudotrials_per_iter_empty,min_pseudotrials_per_iter_social]

In [ ]:
def plot_across_sessions_loop(all_data,decode_types,comparisons,metric='accuracy',figsize=across_sesh_figsize,save=save_figs):
    for comp in comparisons:
        for dt in decode_types:
            if 'empty' in dt:
                pt_filters=[None,min_pseudotrials_per_iter_empty]
            else:
                pt_filters=[None,min_pseudotrials_per_iter_social]
            for pt_filt in pt_filters:
                if pt_filt is not None:
                    filt_type = 'enough_pts'
                else:
                    filt_type = 'all_sessions'
                if save:
                    save_path=os.path.join(across_sesh_figpath,f"{dt}_decoding_{comp}_sessions_{filt_type}.svg")
                else:
                    save_path=None
            
                fig, ax = plot_decoding_trajectory(all_data, decode_type=dt, metric=metric,
                              session_indices=SESSION_SUBSETS[comp], null_type='circular_aware',
                              show_null=False,min_pseudotrials=pt_filt,title_specifier=comp,figsize=figsize,save_path=save_path)
    

In [ ]:
save_figs

In [ ]:
plot_across_sessions_loop(all_pop_data,within_decode_types,comparisons_of_interest,metric='accuracy',figsize=across_sesh_figsize,save=save_figs)

#### Correlating decoding performance vs. score in subsequent partner preference test

In [ ]:
decoding_vs_ppt_path = os.path.join(decoding_path,'ppt_scores_vs_decoding_performance')
decoding_vs_ppt_path

Append PPT score to the decoding performances for the relevant sessions

In [ ]:
identity_across_all_sessions

In [ ]:
tidy_ppt_scores

In [ ]:
def merge_to_ppt_df(neural_data_df,ppt_df,min_pseudotrials=None):
    merged_df = neural_data_df.merge(ppt_df,on=['animal','session_idx'],how='left')
    merged_df = merged_df.dropna(subset=['true_value','ppref_score_cup','ppref_score_chamber']).reset_index(drop=True)
    merged_df = merged_df.drop(columns=['null_mean','null_std','null_values','p_value'])
    if min_pseudotrials is not None:
        merged_df = merged_df[merged_df['mean_train_pts_per_iter']>=min_pseudotrials]
    return merged_df

In [ ]:
identity_decoding_with_ppt_all_sessions = merge_to_ppt_df(identity_across_all_sessions,tidy_ppt_scores,min_pseudotrials=None)
identity_decoding_with_ppt_enough_pts = merge_to_ppt_df(identity_across_all_sessions,tidy_ppt_scores,min_pseudotrials=min_pseudotrials_per_iter_social)

social_location_decoding_with_ppt_all_sessions = merge_to_ppt_df(social_location_across_all_sessions,tidy_ppt_scores,min_pseudotrials=None)
social_location_decoding_with_ppt_enough_pts = merge_to_ppt_df(social_location_across_all_sessions,tidy_ppt_scores,min_pseudotrials=min_pseudotrials_per_iter_social)

empty_location_decoding_with_ppt_all_sessions = merge_to_ppt_df(empty_location_across_all_sessions,tidy_ppt_scores,min_pseudotrials=None)
empty_location_decoding_with_ppt_enough_pts = merge_to_ppt_df(empty_location_across_all_sessions,tidy_ppt_scores,min_pseudotrials=min_pseudotrials_per_iter_empty)

identity_decoding_with_ppt_enough_pts

In [ ]:
if save_csvs:
    identity_decoding_with_ppt_all_sessions.to_csv(os.path.join(decoding_vs_ppt_path,'identity_decoding_vs_ppt_score_all_sessions.csv'),index=False)
    identity_decoding_with_ppt_enough_pts.to_csv(os.path.join(decoding_vs_ppt_path,'identity_decoding_vs_ppt_score_enough_pt_sessions.csv'),index=False)
    social_location_decoding_with_ppt_all_sessions.to_csv(os.path.join(decoding_vs_ppt_path,'social_location_decoding_vs_ppt_score_all_sessions.csv'),index=False)
    social_location_decoding_with_ppt_enough_pts.to_csv(os.path.join(decoding_vs_ppt_path,'social_location_decoding_vs_ppt_score_enough_pt_sessions.csv'),index=False)
    empty_location_decoding_with_ppt_all_sessions.to_csv(os.path.join(decoding_vs_ppt_path,'empty_location_decoding_vs_ppt_score_all_sessions.csv'),index=False)
    empty_location_decoding_with_ppt_enough_pts.to_csv(os.path.join(decoding_vs_ppt_path,'empty_location_decoding_vs_ppt_score_enough_pt_sessions.csv'),index=False)

Stats - pearson correlation coefficient within each session for PPT score vs. decoding performance with Benjamini-Hochberg correction across sessions

In [ ]:
identity_decoding_vs_ppt_all_sessions = correlate_across_within_sessions(identity_decoding_with_ppt_all_sessions,x_col='true_value',y_col='ppref_score_cup',comparison='within_sessions',
                                 session_indices=None,min_pseudotrials=None,min_n=3,alpha=0.05)

identity_decoding_vs_ppt_enough_pts = correlate_across_within_sessions(identity_decoding_with_ppt_all_sessions,x_col='true_value',y_col='ppref_score_cup',comparison='within_sessions',
                                 session_indices=None,min_pseudotrials=min_pseudotrials_per_iter_social,min_n=3,alpha=0.05)

print(identity_decoding_vs_ppt_all_sessions)
identity_decoding_vs_ppt_enough_pts

In [ ]:
if save_csvs:
    identity_decoding_vs_ppt_all_sessions.to_csv(os.path.join(decoding_vs_ppt_path,'identity_decoding_vs_ppt_score_pearson_all_sessions.csv'),index=False)
    identity_decoding_vs_ppt_enough_pts.to_csv(os.path.join(decoding_vs_ppt_path,'identity_decoding_vs_ppt_score_pearson_enough_pts_sessions.csv'),index=False)

In [ ]:
social_location_decoding_vs_ppt_all_sessions = correlate_across_within_sessions(social_location_decoding_with_ppt_all_sessions,x_col='true_value',y_col='ppref_score_cup',comparison='within_sessions',
                                 session_indices=None,min_pseudotrials=None,min_n=3,alpha=0.05)

social_location_decoding_vs_ppt_enough_pts = correlate_across_within_sessions(social_location_decoding_with_ppt_all_sessions,x_col='true_value',y_col='ppref_score_cup',comparison='within_sessions',
                                 session_indices=None,min_pseudotrials=min_pseudotrials_per_iter_social,min_n=3,alpha=0.05)

print(social_location_decoding_vs_ppt_all_sessions)
social_location_decoding_vs_ppt_enough_pts

In [ ]:
if save_csvs:
    social_location_decoding_vs_ppt_all_sessions.to_csv(os.path.join(decoding_vs_ppt_path,'social_location_decoding_vs_ppt_score_pearson_all_sessions.csv'),index=False)
    social_location_decoding_vs_ppt_enough_pts.to_csv(os.path.join(decoding_vs_ppt_path,'social_location_decoding_vs_ppt_score_pearson_enough_pts_sessions.csv'),index=False)

In [ ]:
empty_location_decoding_vs_ppt_all_sessions = correlate_across_within_sessions(empty_location_decoding_with_ppt_all_sessions,x_col='true_value',y_col='ppref_score_cup',comparison='within_sessions',
                                 session_indices=None,min_pseudotrials=None,min_n=3,alpha=0.05)

empty_location_decoding_vs_ppt_enough_pts = correlate_across_within_sessions(empty_location_decoding_with_ppt_all_sessions,x_col='true_value',y_col='ppref_score_cup',comparison='within_sessions',
                                 session_indices=None,min_pseudotrials=min_pseudotrials_per_iter_empty,min_n=3,alpha=0.05)

print(empty_location_decoding_vs_ppt_all_sessions)
empty_location_decoding_vs_ppt_enough_pts

In [ ]:
if save_csvs:
    empty_location_decoding_vs_ppt_all_sessions.to_csv(os.path.join(decoding_vs_ppt_path,'empty_location_decoding_vs_ppt_score_pearson_all_sessions.csv'),index=False)
    empty_location_decoding_vs_ppt_enough_pts.to_csv(os.path.join(decoding_vs_ppt_path,'empty_location_decoding_vs_ppt_score_pearson_enough_pts_sessions.csv'),index=False)

Define plotting functions

In [ ]:
decoding_ppt_fig_path = os.path.join(decoding_vs_ppt_path,'figures')

In [ ]:
def plot_decoding_vs_behavior(df, decode_metric='true_value', behavior_metric='ppref_score_cup',
                               session_indices=None, min_pseudotrials=None,
                               figsize=(6, 5), decode_type_label=None,save_path=None):
    """
    Plot decoding performance vs a behavioral metric.

    """
    all_sessions_requested = session_indices is None
    
    if session_indices is None:
        session_indices = sorted(df['session_idx'].unique())
    
    if min_pseudotrials is not None:
        df = df[df['mean_train_pts_per_iter']>=min_pseudotrials].copy().reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Color by session
    inds_df_list = []
    for s_idx in session_indices:
        s_data = df[df['session_idx'] == s_idx]
        color = SESSION_CONFIG['colors'][s_idx]
        label = SESSION_CONFIG['short_labels'][s_idx]
        ax.scatter(s_data[behavior_metric], s_data[decode_metric],
                  c=color, s=80, alpha=0.7, label=label, edgecolors='white')
        inds_df_list.append(s_data)
    inds_df = pd.concat(inds_df_list).dropna(subset=[behavior_metric, decode_metric])

    # Correlation
    if len(inds_df) > 2:
        r, p = stats.pearsonr(inds_df[behavior_metric], inds_df[decode_metric])
        ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.3f}',
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Regression line
        z = np.polyfit(inds_df[behavior_metric], inds_df[decode_metric], 1)
        p_line = np.poly1d(z)
        x_line = np.linspace(inds_df[behavior_metric].min(), inds_df[behavior_metric].max(), 100)
        ax.plot(x_line, p_line(x_line), 'k--', alpha=0.5, linewidth=1)
    
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)
    
    ax.set_xlabel(behavior_metric.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel(decode_metric.replace('_', ' ').title(), fontsize=12)
    if decode_type_label is not None:
        if len(session_indices)==1:
            ax.set_title(f'{decode_type_label} decoding vs. {behavior_metric}, {SESSION_CONFIG["short_labels"][session_indices[0]]}')
        elif all_sessions_requested:
            ax.set_title(f'{decode_type_label} decoding vs. {behavior_metric}, all timepoints')
        else:
            ax.set_title(f'{decode_type_label} decoding vs. {behavior_metric}')
    else:
        if len(session_indices)==1:
            ax.set_title(f'decoding vs. {behavior_metric}, {SESSION_CONFIG["short_labels"][session_indices[0]]}')
        elif all_sessions_requested:
            ax.set_title(f'decoding vs. {behavior_metric}, all timepoints')
        else:
            ax.set_title(f'decoding vs. {behavior_metric}')

    if min_pseudotrials is not None:
        ax.set_title(ax.get_title() + ', sessions with sufficient SVM training data')
    else:
        ax.set_title(ax.get_title() + ', all sessions')
    
    ax.legend(loc='best')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    return fig, ax

Plot PPT score against decoding performance for each metric, separately for SS, OS presep, and OS postsep

In [ ]:
ppt_vs_behavior_figsize = (6,5)

In [ ]:
ppt_inds_list = [None,[2],[4],[5]]

In [ ]:
def ppt_vs_decoding_plotting_loop(all_sessions_df,session_index_list=ppt_inds_list,min_pseudotrials_type=min_pseudotrials_per_iter_social,
                                  figsize=ppt_vs_behavior_figsize,decode_type_label=None,save=save_figs):
    for idx in session_index_list:
        pt_filters = [None,min_pseudotrials_type]
        for pt_filter in pt_filters:
            if save:
                if pt_filter is not None:
                    filt_type = 'enough_pts'
                else:
                    filt_type = 'all_sessions'
                if idx is not None:
                    sesh_type = SESSION_CONFIG['short_labels'][idx[0]]
                    save_path=os.path.join(decoding_ppt_fig_path,f"ppt_score_vs_{decode_type_label}_decoding_{sesh_type}_{filt_type}.svg")
                else:
                    save_path=os.path.join(decoding_ppt_fig_path,f"ppt_score_vs_{decode_type_label}_decoding_all_timepoints_{filt_type}.svg")
            else:
                save_path=None
            fig, ax = plot_decoding_vs_behavior(all_sessions_df, decode_metric='true_value', behavior_metric='ppref_score_cup',
                                   session_indices=idx, min_pseudotrials=pt_filter,
                                   figsize=figsize,decode_type_label=decode_type_label,save_path=save_path)

In [ ]:
#identity
ppt_vs_decoding_plotting_loop(identity_decoding_with_ppt_all_sessions,session_index_list=ppt_inds_list,min_pseudotrials_type=min_pseudotrials_per_iter_social,
                                  figsize=ppt_vs_behavior_figsize,decode_type_label='identity',save=save_figs)

In [ ]:
#social location
ppt_vs_decoding_plotting_loop(social_location_decoding_with_ppt_all_sessions,session_index_list=ppt_inds_list,min_pseudotrials_type=min_pseudotrials_per_iter_social,
                                  figsize=ppt_vs_behavior_figsize,decode_type_label='social_location',save=save_figs)

In [ ]:
#empty location
ppt_vs_decoding_plotting_loop(empty_location_decoding_with_ppt_all_sessions,session_index_list=ppt_inds_list,min_pseudotrials_type=min_pseudotrials_per_iter_empty,
                                  figsize=ppt_vs_behavior_figsize,decode_type_label='empty_location',save=save_figs)

------------------------

### Mahalanobis distance

In [ ]:
md_path = os.path.join(respath,'mahalanobis_distance')
md_figs_path = os.path.join(md_path,'figures')

Extract metrics into summary dfs and save as .csvs

In [ ]:
md_social_location_df = extract_MD_metric(all_pop_data,metric_name='md_ratio_location_lw',session_indices=SESSION_SUBSETS['social'],animals=None)

In [ ]:
md_identity_df = extract_MD_metric(all_pop_data,metric_name='md_ratio_identity_lw',session_indices=SESSION_SUBSETS['social'],animals=None)

In [ ]:
md_empty_location_df = extract_MD_metric(all_pop_data,metric_name='md_ratio_location_empty1_lw',session_indices=SESSION_SUBSETS['social'],animals=None)

In [ ]:
md_empty_location_df

In [ ]:
if save_csvs:
    md_social_location_df.to_csv(os.path.join(md_path,'md_ratio_social_location_all_sessions.csv'),index=False)
    md_empty_location_df.to_csv(os.path.join(md_path,'md_ratio_empty_location_all_sessions.csv'),index=False)
    md_identity_df.to_csv(os.path.join(md_path,'md_ratio_identity_all_sessions.csv'),index=False)

Plotting

In [ ]:
MD_PLOT_DEFAULTS = {
    'md_ratio_social_location': {
        'label': 'Location (social trials)',
        'color': '#2E86AB',
    },
    'md_ratio_identity': {
        'label': 'Identity',
        'color': '#A23B72',
    },
    'md_ratio_location_empty1': {
        'label': 'Location (empty trial 1)',
        'color': '#545454',
    },
}

In [ ]:
#defining dictionaries for plotting
all_three_MDs = {
    'md_ratio_social_location': md_social_location_df,
    'md_ratio_identity': md_identity_df,
    'md_ratio_location_empty1': md_empty_location_df
}

id_vs_loc_MDs = {
    'md_ratio_social_location': md_social_location_df,
    'md_ratio_identity': md_identity_df
}

loc_vs_loc_MDs = {
    'md_ratio_social_location': md_social_location_df,
    'md_ratio_location_empty1': md_empty_location_df
}

id_only_MDs = {
    'md_ratio_identity': md_identity_df
}

social_loc_only_MDs = {
    'md_ratio_social_location': md_social_location_df,
}

empty_loc_only_MDs = {
    'md_ratio_location_empty1': md_empty_location_df,
}


In [ ]:
md_dfs = id_vs_loc_MDs

In [ ]:
session_sets = [set(df['session_idx'].unique()) for df in md_dfs.values()]
session_indices = sorted(set.intersection(*session_sets))
session_indices

In [ ]:
offset_x=True
n_metrics = len(md_dfs)
if offset_x and n_metrics > 1:
    offset_step = 0.06
    offsets = np.linspace(-(n_metrics - 1) / 2 * offset_step,
                            (n_metrics - 1) / 2 * offset_step,
                            n_metrics)
offsets

In [ ]:
def plot_MD_trajectory(md_dfs, session_indices=None, offset_x=False,
                       ylabel='Mahalanobis Distance Ratio',
                       title=None, figsize=(10, 6), save_path=None):
    """
    Plot 1–3 Mahalanobis distance metrics across sessions, with individual
    animal trajectories and group mean ± SEM for each.

    Parameters
    ----------
    md_dfs : dict
        Keys are metric names (e.g. 'md_ratio_location_lw'), values are
        DataFrames from extract_MD_metric(). Pass 1, 2, or 3 entries.
    session_indices : list[int] or None
        Sessions to plot. If None, uses sessions present in all DataFrames.
    """
    if session_indices is None:
        session_sets = [set(df['session_idx'].unique()) for df in md_dfs.values()]
        session_indices = sorted(set.intersection(*session_sets))

    x_positions = list(range(len(session_indices)))
    x_labels = [SESSION_CONFIG['short_labels'][i] for i in session_indices]
    x_colors = [SESSION_CONFIG['colors'][i] for i in session_indices]

    fig, ax = plt.subplots(figsize=figsize)

    # Compute offsets so metrics are centered at each x position
    n_metrics = len(md_dfs)
    if offset_x and n_metrics > 1:
        offset_step = 0.06
        offsets = np.linspace(-(n_metrics - 1) / 2 * offset_step,
                               (n_metrics - 1) / 2 * offset_step,
                               n_metrics)
    else:
        offsets = np.zeros(n_metrics)

    for (metric_name, df), dx in zip(md_dfs.items(), offsets):
        defaults = MD_PLOT_DEFAULTS.get(metric_name, {})
        color = defaults.get('color', 'black')
        label = defaults.get('label', metric_name)

        # Individual animal trajectories
        for animal in df['animal'].unique():
            animal_data = df[df['animal'] == animal].set_index('session_idx')
            y_values = [animal_data.loc[s, 'value'] if s in animal_data.index else np.nan
                        for s in session_indices]
            x_shifted = [x + dx for x in x_positions]
            ax.plot(x_shifted, y_values, 'o-', color=color, alpha=0.3,
                    markersize=6, linewidth=1)

        # Group mean ± SEM
        means, sems = [], []
        for s_idx in session_indices:
            vals = df[df['session_idx'] == s_idx]['value'].dropna()
            means.append(vals.mean())
            sems.append(vals.sem() if len(vals) > 1 else 0)

        x_shifted = [x + dx for x in x_positions]
        ax.errorbar(x_shifted, means, yerr=sems, fmt='s-', color=color,
                    markersize=10, linewidth=2.5, capsize=5, capthick=2,
                    markeredgecolor='black', markeredgewidth=1, zorder=10,
                    label=label)

    # Formatting
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, fontsize=11)
    for ticklabel, color in zip(ax.get_xticklabels(), x_colors):
        ticklabel.set_color(color)
        ticklabel.set_fontweight('bold')

    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xlabel('Timepoint', fontsize=12)
    ax.legend(loc='best')

    if title is not None:
        ax.set_title(title, fontsize=14)

    ax.axhline(0, color='red', linestyle='--', alpha=1, linewidth=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()
    return fig, ax

In [ ]:
ylabel = "Mahalanobis distance ratio: log(MDdiff/MDsame)"

In [ ]:
md_figs_path

In [ ]:
if save_figs:
    MD_across_sesh_savepath=os.path.join(md_figs_path,'md_ratio_across_all_timepoints_all_variables.svg')
else:
    MD_across_sesh_savepath=None
    
fig, ax = plot_MD_trajectory(all_three_MDs, session_indices=SESSION_SUBSETS['social'],offset_x=False,ylabel=ylabel,
                             figsize=(10,6),title='Mahalanobis distance ratio over all timepoints (all variables)',save_path=MD_across_sesh_savepath)

Plot and optionally save 1) social location MDR vs. identity MDR and 2) social location vs. empty location MDR at all interesting timepoint comparisons

In [ ]:
comparisons_of_interest

In [ ]:
md_comparisons_figsize = (7,6)

In [ ]:
def plot_comparisons_loop(mds_dict,sesh_comparisons,variable_comp_name=None,ylabel=ylabel,figsize=md_comparisons_figsize,save=save_figs):
    for comp in sesh_comparisons:
        title = f"Mahalanobis distance ratio across {comp} timepoints"
        if save:
            savepath=os.path.join(md_figs_path,f'{variable_comp_name}_across_{comp}_timepoints.svg')
        else:
            savepath=None
        fig, ax = plot_MD_trajectory(mds_dict,session_indices=SESSION_SUBSETS[comp],offset_x=False,ylabel=ylabel,title=title,
                                     figsize=figsize,save_path=savepath)

In [ ]:
#identity vs. location during social trials
plot_comparisons_loop(id_vs_loc_MDs,comparisons_of_interest,variable_comp_name='id_vs_social_location_MD_ratio')

In [ ]:
#location during empty trial 1 vs. location during social trials
plot_comparisons_loop(loc_vs_loc_MDs,comparisons_of_interest,variable_comp_name='empty_location_vs_social_location_MD_ratio')

Stats

1) one-way t-test vs. null at each time point -- are all MD types significantly above 0? -- BH correction for multiple comparisons

In [ ]:
one_way_vs_null_dict ={}

for label, df in all_three_MDs.items():
    metric_res_list = []
    for sesh_idx in SESSION_SUBSETS['social']:
        res = one_sample_vs_chance(df,sesh_idx,chance_level=0,test='ttest',alternative='greater')
        if res is not None:
            metric_res_list.append(res)
    metric_res_df = pd.DataFrame(metric_res_list)
    metric_res_df['metric']=label
    #BH correction
    valid_mask = metric_res_df['p_value'].notna()
    if valid_mask.sum() > 0:
        rejected, pvals_corr, _, _ = multipletests(
            metric_res_df.loc[valid_mask, 'p_value'], alpha=0.05, method='fdr_bh')
        metric_res_df['p_corrected'] = np.nan
        metric_res_df.loc[valid_mask, 'p_corrected'] = pvals_corr
        metric_res_df['significant'] = False
        metric_res_df.loc[valid_mask, 'significant'] = rejected
    one_way_vs_null_dict[label]=metric_res_df

In [ ]:
one_way_vs_null_dict['md_ratio_identity']

In [ ]:
one_way_vs_null_dict['md_ratio_social_location']

In [ ]:
one_way_vs_null_dict['md_ratio_location_empty1']

In [ ]:
if save_csvs:
    one_way_vs_null_dict['md_ratio_identity'].to_csv(os.path.join(md_path,"one_way_vs_null_res_MD_identity_all_timepoints.csv"),index=False)
    one_way_vs_null_dict['md_ratio_social_location'].to_csv(os.path.join(md_path,"one_way_vs_null_res_MD_social_location_all_timepoints.csv"),index=False)
    one_way_vs_null_dict['md_ratio_location_empty1'].to_csv(os.path.join(md_path,"one_way_vs_null_res_MD_empty_location_all_timepoints.csv"),index=False)
    

2) Wilcoxon within each timepoint -- is MD location larger than MD identity (or is MD location during social trials different than MD location during empty trial 1) at each time point (w/ BH correction for multiple comparisons)

In [ ]:
loc_vs_id_per_session = compare_metrics_within_sessions(md_social_location_df,md_identity_df,label_a='social_location',label_b='identity',
                                                       session_indices=None,test='wilcoxon',alternative='two-sided',alpha=0.05)
loc_vs_loc_per_session = compare_metrics_within_sessions(md_social_location_df,md_empty_location_df,label_a='social_location',label_b='empty_location',
                                                       session_indices=None,test='wilcoxon',alternative='two-sided',alpha=0.05)

In [ ]:
loc_vs_id_per_session

In [ ]:
#save out .csvs
if save_csvs:
    loc_vs_id_per_session.to_csv(os.path.join(md_path,"social_location_vs_identity_md_sig_per_timepoint.csv"))
    loc_vs_loc_per_session.to_csv(os.path.join(md_path,"social_location_vs_empty_location_md_sig_per_timepoint.csv"))

3) paired comparison at each interesting comparison timepoint within MD types w BH correction for multiple comparisons --> wilcoxon bw md ratio values within each variable at various timepoint pairs with benjamini-hochberg correction for multiple comparisons

In [ ]:
paired_comp_res_dict = {}

for label, df in all_three_MDs.items():
    res_list = []
    for comp in comparisons_of_interest:
        res = paired_comparison(df,session_indices=SESSION_SUBSETS[comp],test='wilcoxon',alternative='two-sided')
        if res is None:
            continue
        del res['data']
        res_list.append(res)
    res_df = pd.DataFrame(res_list)
    res_df['metric']=label

    valid_mask = res_df['p_value'].notna()
    if valid_mask.sum() > 0:
        rejected, pvals_corr, _, _ = multipletests(
            res_df.loc[valid_mask, 'p_value'], alpha=0.05, method='fdr_bh')
        res_df['p_corrected'] = np.nan
        res_df.loc[valid_mask, 'p_corrected'] = pvals_corr
        res_df['significant'] = False
        res_df.loc[valid_mask, 'significant'] = rejected

    paired_comp_res_dict[label]=res_df

In [ ]:
paired_comp_res_dict['md_ratio_identity']

In [ ]:
if save_csvs:
    paired_comp_res_dict['md_ratio_identity'].to_csv(os.path.join(md_path,"paired_comparisons_sig_MD_identity_all_timepoints.csv"),index=False)
    paired_comp_res_dict['md_ratio_social_location'].to_csv(os.path.join(md_path,"paired_comparisons_sig_MD_social_location_all_timepoints.csv"),index=False)
    paired_comp_res_dict['md_ratio_location_empty1'].to_csv(os.path.join(md_path,"paired_comparisons_sig_MD_empty_location_all_timepoints.csv"),index=False)

4) Does any MD scale at any timepoint with PPT score? --> Pearson correlation bw MD ratio for each variable and PPT score (at each imaging timepoint with a subsequent PPT)

In [ ]:
tidy_ppt_scores

In [ ]:
md_social_location_df

In [ ]:
ppt_md_comp_dict = {}
for label, df in all_three_MDs.items():
    ppt_md_comp_dict[label]=merge_ppt_scores_neural_data(df,tidy_ppt_scores)

In [ ]:
ppt_md_comp_dict['md_ratio_identity']

In [ ]:
ppt_md_correlation_res_dict = {}
for label, df in ppt_md_comp_dict.items():
    ppt_md_correlation_res_dict[label]=correlate_across_within_timepoints(df,x_col='ppref_score_cup',y_col='value',comparison='within_sessions',
                                                                         session_indices=None,min_n=3,addl_info=label,alpha=0.05)

In [ ]:
ppt_md_correlation_res_dict['md_ratio_identity']

In [ ]:
#add .csv saveouts

In [ ]:
if save_csvs:
    ppt_md_correlation_res_dict['md_ratio_identity'].to_csv(os.path.join(md_path,"ppt_score_md_ratio_identity_corr_all_timepoints.csv"),index=False)
    ppt_md_correlation_res_dict['md_ratio_location_empty1'].to_csv(os.path.join(md_path,"ppt_score_md_ratio_empty_location_corr_all_timepoints.csv"),index=False)
    ppt_md_correlation_res_dict['md_ratio_social_location'].to_csv(os.path.join(md_path,"ppt_score_md_ratio_social_location_corr_all_timepoints.csv"),index=False)

------------